# cool-title-here

**Group Members:** Aaron Go, John Alonzo, Sean Olores, Sean Cardeno 

**Deadline:** March 21, 2026 (Saturday), 8:00 AM

**Task:** Predict **the Challenge Rating of a monster** based on **the monster's features**

**Dataset Source:** [DnD 5e Monsters](https://www.kaggle.com/datasets/mrpantherson/dnd-5e-monsters/data)

**Justification:** In Dungeons & Dragons, *Challenge Rating* is a mathematical estimation of a monster's combat efficiency. A model that learns to predict this value from raw stats allows Game Masters to instantly balance custom *homebrew* content without manually calculating complex design formulas.


## Section 1: Data Preparation

In this section, we will:
- Load the dataset
- Inspect the data structure and basic statistics
- Handle missing values and data cleaning
- Identify data types and any inconsistencies

### Setup: imports and initial dataset load
We'll begin by importing necessary libraries and reading the CSV into a pandas DataFrame.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/dnd_monsters_completed.csv') 
print("Original shape of data:", df.shape)

Original shape of data: (738, 17)


In [2]:
# Data Cleaning: convert CR, fix entries, and map sizes

# convert fraction strings in challenge rating column to floats
def convert_cr_to_float(cr_val):
    if pd.isna(cr_val):  # If the cell is completely empty, leave it
        return cr_val
    
    cr_str = str(cr_val).strip()
    
    if '/' in cr_str:
        numerator, denominator = cr_str.split('/')
        return float(numerator) / float(denominator)
    
    return float(cr_str)

df['cr'] = df['cr'].apply(convert_cr_to_float)

# ALDANI fix, aldani CR is null in row 22, supplemented with data from 5e.tools
df.loc[df['name'].str.lower() == 'aldani', 'cr'] = 1.0

# ordinal encoding the size
df['size'] = df['size'].astype(str).str.capitalize().str.strip()
size_mapping = {'Tiny': 0, 'Small': 1, 'Medium': 2, 'Large': 3, 'Huge': 4, 'Gargantuan': 5}
df['size'] = df['size'].map(size_mapping)


In [3]:
# Review results after initial cleaning
missing_cr_count = df['cr'].isna().sum()
print(f"Monsters missing CR: {missing_cr_count}")

print("New shape of data:", df.shape)
df.head()

Monsters missing CR: 0
New shape of data: (738, 17)


,name,url,cr,type,size,ac,hp,speed,align,legendary,source,str,dex,con,int,wis,cha
0,aarakocra,https://www.aidedd.org/dnd/monstres.php?vo=aar...,0.25,humanoid (aarakocra),2,12,13,fly,neutral good,False,Monster Manual (BR),10.0,14.0,10.0,11.0,12.0,11.0
1,abjurer,NaN,9.00,humanoid (any race),2,12,84,NaN,any alignment,False,Volo's Guide to Monsters,NaN,NaN,NaN,NaN,NaN,NaN
2,aboleth,https://www.aidedd.org/dnd/monstres.php?vo=abo...,10.00,aberration,3,17,135,swim,lawful evil,True,Monster Manual (SRD),21.0,9.0,15.0,18.0,15.0,18.0
3,abominable-yeti,NaN,9.00,monstrosity,4,15,137,NaN,chaotic evil,False,Monster Manual,24.0,10.0,22.0,9.0,13.0,9.0
4,acererak,NaN,23.00,undead,2,21,285,NaN,neutral evil,True,Adventures (Tomb of Annihilation),13.0,16.0,20.0,27.0,21.0,20.0


In [4]:
# Calculate the total missing values for every single column
missing_data = df.isna().sum()

# Filter to ONLY show columns that actually have missing data (greater than 0)
print("Columns with missing data before fixing:")
print(missing_data[missing_data > 0])


Columns with missing data before fixing:
url      337
speed    490
str       43
dex       43
con       43
int       43
wis       43
cha       43
dtype: int64


### Cleaning workflow overview
The operations below are executed sequentially within Section 1:

1. Drop irrelevant metadata columns (e.g. `url`, `source`, `align`).
2. Fill missing text values (speeds become `'None'`).
3. Remove rows lacking any core ability score.
4. Extract movement-type flags (`has_fly`, `has_swim`).
5. Flag shapechangers and one-hot encode the base `type`.
6. Save a cleaned checkpoint for later use.


In [5]:
# Drop columns that don't help prediction
df = df.drop(columns=['url', 'source', 'align'], errors='ignore')

# Fill text NAs: convert missing speeds to 'None'
df['speed'] = df['speed'].fillna('None')


### Additional Cleaning: Stats and Movement Features

The next steps remove incomplete stat records and extract whether monsters have specialized movement types from the `speed` field.

In [6]:
# Drop rows with missing core ability scores
stat_columns = ['str', 'dex', 'con', 'int', 'wis', 'cha']
df = df.dropna(subset=stat_columns)

# Create binary features for specialized movement (fly/swim)
df['has_fly'] = df['speed'].str.contains('fly', case=False, na=False).astype(int)
df['has_swim'] = df['speed'].str.contains('swim', case=False, na=False).astype(int)

df = df.drop(columns=['speed'])

### Feature Encoding from `type` Column
This section handles shapechanger detection and converts the textual `type` field into one-hot numerical features.

In [7]:
# Flag shapechangers and prepare base-type encoding
df['is_shapechanger'] = df['type'].str.contains('shapechanger', case=False, na=False).astype(int)

# Extract the base type (text before any parenthesis) for one-hot encoding
# e.g. "humanoid (aarakocra)" -> "humanoid"
df['base_type'] = df['type'].str.extract(r'^([a-zA-Z\-]+)')[0].str.strip().str.lower()

print("Unique base types:", df['base_type'].unique())

# One-hot encode categorical 'base_type' values
type_dummies = pd.get_dummies(df['base_type'], prefix='type')
df = pd.concat([df, type_dummies], axis=1)
df = df.drop(columns=['type', 'base_type'])

# Save cleaned dataframe as a checkpoint for downstream analysis
print("Final shape:", df.shape)
print("Any remaining nulls:\n", df.isna().sum()[df.isna().sum() > 0])
df.to_csv('../data/processed/cleaned_monsters.csv', index=False)
print("\nSection 1 Complete! Data exported to cleaned_monsters.csv")

Unique base types: <StringArray>
[   'humanoid',  'aberration', 'monstrosity',      'undead',      'dragon',
        'ooze',   'elemental',       'fiend',       'beast',   'construct',
         'fey',       'plant',       'giant',   'celestial',       'swarm']
Length: 15, dtype: str
Final shape: (695, 30)
Any remaining nulls:
 Series([], dtype: int64)

Section 1 Complete! Data exported to cleaned_monsters.csv
